| Strategy | Logic | Best For |
| :--- | :--- | :--- |
| **Grid Search** | Exhaustive search through every possible combination. | Very small search spaces. |
| **Random Search** | Samples randomly from the distribution. | Surprisingly effective for high-dimensional spaces. |
| **Bayesian (TPE/GP)** | Uses past results to predict which parameters will perform better. | Expensive-to-train models (Deep Learning). |

Optuna Paper - https://arxiv.org/pdf/1907.10902

Bayesian Optimization (TPE) Paper - https://arxiv.org/pdf/2304.11127


In [1]:
# !pip install optuna

In [2]:
# Import necessary libraries
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the Pima Indian Diabetes dataset from sklearn
# Note: Scikit-learn's built-in 'load_diabetes' is a regression dataset.
# We will load the actual diabetes dataset from an external source
import pandas as pd

# Load the Pima Indian Diabetes dataset (from UCI repository)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

# Load the dataset
df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [4]:
import numpy as np

# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# Impute the missing values with the mean of the respective column
df.fillna(df.median(), inplace=True)

# Check if there are any remaining missing values
print(df.isnull().sum())


Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [5]:
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional: Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')


Training set shape: (537, 8)
Test set shape: (231, 8)


In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize

In [7]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters

[I 2026-05-06 15:56:49,344] A new study created in memory with name: no-name-e66349a8-9d0d-4a15-9f55-b20e629b5318
[I 2026-05-06 15:56:50,059] Trial 0 finished with value: 0.7579143389199254 and parameters: {'n_estimators': 137, 'max_depth': 3}. Best is trial 0 with value: 0.7579143389199254.
[I 2026-05-06 15:56:50,737] Trial 1 finished with value: 0.7765363128491621 and parameters: {'n_estimators': 138, 'max_depth': 7}. Best is trial 1 with value: 0.7765363128491621.
[I 2026-05-06 15:56:50,997] Trial 2 finished with value: 0.7746741154562384 and parameters: {'n_estimators': 60, 'max_depth': 11}. Best is trial 1 with value: 0.7765363128491621.
[I 2026-05-06 15:56:51,674] Trial 3 finished with value: 0.7597765363128491 and parameters: {'n_estimators': 198, 'max_depth': 6}. Best is trial 1 with value: 0.7765363128491621.
[I 2026-05-06 15:56:51,906] Trial 4 finished with value: 0.7579143389199254 and parameters: {'n_estimators': 82, 'max_depth': 6}. Best is trial 1 with value: 0.7765363128

In [8]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7802607076350094
Best hyperparameters: {'n_estimators': 122, 'max_depth': 7}


In [9]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.75


## Samplers in Optuna

In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize


In [11]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters

[I 2026-05-06 15:57:12,377] A new study created in memory with name: no-name-9f1a3b06-a002-4d9e-8e75-75cb24227f25
[I 2026-05-06 15:57:12,936] Trial 0 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 184, 'max_depth': 4}. Best is trial 0 with value: 0.7635009310986964.
[I 2026-05-06 15:57:13,497] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 188, 'max_depth': 17}. Best is trial 1 with value: 0.7672253258845437.
[I 2026-05-06 15:57:13,924] Trial 2 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 137, 'max_depth': 13}. Best is trial 1 with value: 0.7672253258845437.
[I 2026-05-06 15:57:14,259] Trial 3 finished with value: 0.7597765363128491 and parameters: {'n_estimators': 107, 'max_depth': 15}. Best is trial 1 with value: 0.7672253258845437.
[I 2026-05-06 15:57:14,605] Trial 4 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 99, 'max_depth': 11}. Best is trial 1 with value: 0.767225

In [12]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7802607076350094
Best hyperparameters: {'n_estimators': 91, 'max_depth': 7}


In [13]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')

Test Accuracy with best hyperparameters: 0.75


In [14]:
search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [5, 10, 15, 20]
}

In [15]:
# Create a study and optimize it using GridSampler
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective)

[I 2026-05-06 15:57:32,030] A new study created in memory with name: no-name-8e6f251b-9ddb-4661-86df-8a3f98f393ab
[I 2026-05-06 15:57:32,367] Trial 0 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7653631284916201.
[I 2026-05-06 15:57:32,860] Trial 1 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7653631284916201.
[I 2026-05-06 15:57:33,035] Trial 2 finished with value: 0.7616387337057727 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 0 with value: 0.7653631284916201.
[I 2026-05-06 15:57:33,363] Trial 3 finished with value: 0.7616387337057727 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 0 with value: 0.7653631284916201.
[I 2026-05-06 15:57:33,670] Trial 4 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 0 with value: 0.765363

In [16]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7746741154562384
Best hyperparameters: {'n_estimators': 50, 'max_depth': 10}


In [17]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.76


## Optuna Visualizations

In [18]:
# For visualizations
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [19]:
# 1. Optimization History
fig = optuna.visualization.plot_optimization_history(study)
fig.write_html("optimization_history.html")

In [20]:
# 2. Parallel Coordinates Plot
plot_parallel_coordinate(study).write_html("Parallel Coordinates Plot.html")

In [21]:
# 3. Slice Plot
plot_slice(study).write_html("Slice Plot.html")

In [22]:
# 4. Contour Plot
plot_contour(study).write_html("Contour Plot.html")

In [23]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

## Optimizing Multiple ML Models

In [24]:
# Importing the required libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [25]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [26]:
# Create a study and optimize it using CmaEsSampler
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2026-05-06 15:57:41,499] A new study created in memory with name: no-name-828df93a-24a0-4a12-a2ac-4d2c457381f4
[I 2026-05-06 15:57:43,359] Trial 0 finished with value: 0.7690875232774674 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 256, 'learning_rate': 0.016682240878155473, 'max_depth': 17, 'min_samples_split': 6, 'min_samples_leaf': 7}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-05-06 15:57:44,503] Trial 1 finished with value: 0.7057728119180634 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 112, 'learning_rate': 0.021456716878045304, 'max_depth': 19, 'min_samples_split': 3, 'min_samples_leaf': 1}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-05-06 15:57:46,949] Trial 2 finished with value: 0.7299813780260708 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 202, 'learning_rate': 0.02636382206842232, 'max_depth': 10, 'min_samples_split': 4, 'min_samples_leaf': 2}. Best is trial 0 with value: 0.76908

In [27]:
# Retrieve the best trial
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'SVM', 'C': 0.11001193121165713, 'kernel': 'linear', 'gamma': 'auto'}
Best trial accuracy: 0.7858472998137803


In [28]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.769088,2026-05-06 15:57:41.500462,2026-05-06 15:57:43.359536,0 days 00:00:01.859074,NaN,NaN,GradientBoosting,NaN,NaN,0.016682,17.0,7.0,6.0,256.0,COMPLETE
1,1,0.705773,2026-05-06 15:57:43.360674,2026-05-06 15:57:44.503525,0 days 00:00:01.142851,NaN,NaN,GradientBoosting,NaN,NaN,0.021457,19.0,1.0,3.0,112.0,COMPLETE
2,2,0.729981,2026-05-06 15:57:44.504323,2026-05-06 15:57:46.949061,0 days 00:00:02.444738,NaN,NaN,GradientBoosting,NaN,NaN,0.026364,10.0,2.0,4.0,202.0,COMPLETE
3,3,0.759777,2026-05-06 15:57:46.949861,2026-05-06 15:57:47.603003,0 days 00:00:00.653142,NaN,True,RandomForest,NaN,NaN,NaN,11.0,7.0,7.0,198.0,COMPLETE
4,4,0.739292,2026-05-06 15:57:47.604045,2026-05-06 15:57:48.503205,0 days 00:00:00.899160,NaN,NaN,GradientBoosting,NaN,NaN,0.010007,8.0,4.0,2.0,121.0,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.782123,2026-05-06 15:58:07.040344,2026-05-06 15:58:07.060678,0 days 00:00:00.020334,0.136420,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
96,96,0.767225,2026-05-06 15:58:07.061496,2026-05-06 15:58:07.088362,0 days 00:00:00.026866,0.119526,NaN,SVM,scale,sigmoid,NaN,NaN,NaN,NaN,NaN,COMPLETE
97,97,0.780261,2026-05-06 15:58:07.089247,2026-05-06 15:58:07.107840,0 days 00:00:00.018593,0.278319,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
98,98,0.757914,2026-05-06 15:58:07.109005,2026-05-06 15:58:07.544501,0 days 00:00:00.435496,NaN,True,RandomForest,NaN,NaN,NaN,17.0,9.0,3.0,129.0,COMPLETE


In [29]:
study.trials_dataframe()['params_classifier'].value_counts()

params_classifier
SVM                 76
GradientBoosting    14
RandomForest        10
Name: count, dtype: int64

In [30]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

params_classifier
GradientBoosting    0.743815
RandomForest        0.761080
SVM                 0.768058
Name: value, dtype: float64

In [31]:
# 1. Optimization History
plot_optimization_history(study).write_html('data2/Optimization History.html')

In [32]:
# 3. Slice Plot
plot_slice(study).write_html('data2/Slice Plot.html')

In [33]:
# 5. Hyperparameter Importance
plot_param_importances(study).write_html('data2/Hyperparameter Importance.html')

In [36]:
import optuna
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
import numpy as np

# Load the Iris dataset
X, y = load_iris(return_X_y=True)

# Split the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the objective function for XGBoost
def objective(trial):
    # Hyperparameter search space
    param = {
        'verbosity': 0,
        'objective': 'multi:softprob',
        'num_class': 3,
        'eval_metric': 'mlogloss',  # Ensure that the eval_metric is specified here
        'booster': 'gbtree',
        'lambda': trial.suggest_float('lambda', 1e-8, 1.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-8, 1.0, log=True),
        'eta': trial.suggest_float('eta', 0.01, 0.3),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'n_estimators': 300,
    }

    # Create DMatrix for XGBoost
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_test, label=y_test)

    # Define a pruning callback based on evaluation metrics
    pruning_callback = optuna.integration.XGBoostPruningCallback(trial, "eval-mlogloss")  # Match the metric name in the evals list

    # Train the model
    bst = xgb.train(
        param,
        dtrain,
        num_boost_round=300,
        evals=[(dtrain, "train"), (dtest, "eval")],  # Ensure the eval datasets and names are specified
        early_stopping_rounds=30,
        callbacks=[pruning_callback]
    )

    # Predict on the test set
    preds = bst.predict(dtest)
    best_preds = [int(np.argmax(line)) for line in preds]

    # Return accuracy as the objective value
    accuracy = accuracy_score(y_test, best_preds)
    return accuracy

# Create a study with pruning
study = optuna.create_study(direction='maximize', pruner=optuna.pruners.SuccessiveHalvingPruner())
study.optimize(objective, n_trials=50)

# Output the best trial
print(f"Best trial: {study.best_trial.params}")
print(f"Best accuracy: {study.best_value}")

[I 2026-05-06 16:08:38,399] A new study created in memory with name: no-name-c8937418-23dc-4557-aba2-c3a030d84eb5


[0]	train-mlogloss:0.99754	eval-mlogloss:0.99411
[1]	train-mlogloss:0.78284	eval-mlogloss:0.76826
[2]	train-mlogloss:0.63049	eval-mlogloss:0.60956
[3]	train-mlogloss:0.54405	eval-mlogloss:0.51168
[4]	train-mlogloss:0.47797	eval-mlogloss:0.43481
[5]	train-mlogloss:0.40372	eval-mlogloss:0.35277
[6]	train-mlogloss:0.36165	eval-mlogloss:0.30710
[7]	train-mlogloss:0.34349	eval-mlogloss:0.29028
[8]	train-mlogloss:0.30744	eval-mlogloss:0.24905
[9]	train-mlogloss:0.28055	eval-mlogloss:0.22117
[10]	train-mlogloss:0.26885	eval-mlogloss:0.20788
[11]	train-mlogloss:0.26039	eval-mlogloss:0.19926
[12]	train-mlogloss:0.25829	eval-mlogloss:0.19630
[13]	train-mlogloss:0.25171	eval-mlogloss:0.18853
[14]	train-mlogloss:0.25133	eval-mlogloss:0.18701
[15]	train-mlogloss:0.25081	eval-mlogloss:0.18626
[16]	train-mlogloss:0.25037	eval-mlogloss:0.18377
[17]	train-mlogloss:0.25013	eval-mlogloss:0.18306
[18]	train-mlogloss:0.24997	eval-mlogloss:0.18369
[19]	train-mlogloss:0.24946	eval-mlogloss:0.18379
[20]	train

[I 2026-05-06 16:08:47,222] Trial 0 finished with value: 1.0 and parameters: {'lambda': 3.3040973683415064e-06, 'alpha': 3.9530011973898634e-05, 'eta': 0.2098190616012191, 'gamma': 0.003439146899030436, 'max_depth': 9, 'min_child_weight': 10, 'subsample': 0.8516066499538626, 'colsample_bytree': 0.49771044637713563}. Best is trial 0 with value: 1.0.


[0]	train-mlogloss:1.02166	eval-mlogloss:1.03172
[1]	train-mlogloss:0.85909	eval-mlogloss:0.86165
[2]	train-mlogloss:0.73217	eval-mlogloss:0.72359
[3]	train-mlogloss:0.65465	eval-mlogloss:0.63761
[4]	train-mlogloss:0.58139	eval-mlogloss:0.56366
[5]	train-mlogloss:0.50349	eval-mlogloss:0.47983
[6]	train-mlogloss:0.45785	eval-mlogloss:0.43388
[7]	train-mlogloss:0.42639	eval-mlogloss:0.39961
[8]	train-mlogloss:0.37788	eval-mlogloss:0.34638
[9]	train-mlogloss:0.34854	eval-mlogloss:0.31745
[10]	train-mlogloss:0.31431	eval-mlogloss:0.27902
[11]	train-mlogloss:0.28806	eval-mlogloss:0.24757
[12]	train-mlogloss:0.28053	eval-mlogloss:0.24192
[13]	train-mlogloss:0.25472	eval-mlogloss:0.21265
[14]	train-mlogloss:0.24875	eval-mlogloss:0.20802
[15]	train-mlogloss:0.23691	eval-mlogloss:0.19505
[16]	train-mlogloss:0.22610	eval-mlogloss:0.18316
[17]	train-mlogloss:0.21318	eval-mlogloss:0.17057
[18]	train-mlogloss:0.20547	eval-mlogloss:0.16272
[19]	train-mlogloss:0.19795	eval-mlogloss:0.15542
[20]	train

[I 2026-05-06 16:08:54,005] Trial 1 finished with value: 1.0 and parameters: {'lambda': 0.24341885909454716, 'alpha': 0.3294730773240702, 'eta': 0.15450634511136713, 'gamma': 4.677876118487861e-07, 'max_depth': 3, 'min_child_weight': 1, 'subsample': 0.6165517486065253, 'colsample_bytree': 0.46897161669517784}. Best is trial 0 with value: 1.0.


...

[0]	train-mlogloss:0.86446	eval-mlogloss:0.84358
[1]	train-mlogloss:0.69931	eval-mlogloss:0.66609


[I 2026-05-06 16:09:37,277] Trial 48 pruned. Trial was pruned at iteration 2.


[0]	train-mlogloss:1.08345	eval-mlogloss:1.08447
[1]	train-mlogloss:1.04607	eval-mlogloss:1.04542
[2]	train-mlogloss:1.01057	eval-mlogloss:1.00869
[3]	train-mlogloss:0.98449	eval-mlogloss:0.97932
[4]	train-mlogloss:0.95750	eval-mlogloss:0.95013
[5]	train-mlogloss:0.92594	eval-mlogloss:0.91680
[6]	train-mlogloss:0.90271	eval-mlogloss:0.89383
[7]	train-mlogloss:0.88564	eval-mlogloss:0.87749
[8]	train-mlogloss:0.85757	eval-mlogloss:0.84725
[9]	train-mlogloss:0.83711	eval-mlogloss:0.82611
[10]	train-mlogloss:0.81138	eval-mlogloss:0.79931
[11]	train-mlogloss:0.78909	eval-mlogloss:0.77666
[12]	train-mlogloss:0.78061	eval-mlogloss:0.76577
[13]	train-mlogloss:0.75753	eval-mlogloss:0.74055
[14]	train-mlogloss:0.74989	eval-mlogloss:0.73412
[15]	train-mlogloss:0.73706	eval-mlogloss:0.72183
[16]	train-mlogloss:0.72315	eval-mlogloss:0.70707
[17]	train-mlogloss:0.70691	eval-mlogloss:0.69055
[18]	train-mlogloss:0.69545	eval-mlogloss:0.67927
[19]	train-mlogloss:0.68460	eval-mlogloss:0.66853
[20]	train

[I 2026-05-06 16:09:38,224] Trial 49 pruned. Trial was pruned at iteration 32.


Best trial: {'lambda': 3.3040973683415064e-06, 'alpha': 3.9530011973898634e-05, 'eta': 0.2098190616012191, 'gamma': 0.003439146899030436, 'max_depth': 9, 'min_child_weight': 10, 'subsample': 0.8516066499538626, 'colsample_bytree': 0.49771044637713563}
Best accuracy: 1.0


In [35]:
 # pip install optuna-integration[xgboost]

In [37]:
from optuna.visualization import plot_intermediate_values

# 1. Plot intermediate values during the trials
plot_intermediate_values(study).write_html('data2/intermediate values during the trials.html')